# Volatility Forecasting & Options Analytics — Research Demo

This notebook walks through the full pipeline: **implied-volatility surface → Monte Carlo pricing
with variance reduction → ML realized-vol forecasting vs. baselines → implied-vs-forecast
richness view**.

Data: SPY (S&P 500 ETF) via yfinance, cached on disk after the first run.
All CV predictions are strictly out-of-sample — no lookahead.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from volforecast.data.loader import YFinanceSource
from volforecast.pricing.black_scholes import bsm_price
from volforecast.montecarlo.gbm import mc_price
from volforecast.montecarlo.variance_reduction import mc_price_antithetic, mc_price_control_variate
from volforecast.surface.build_surface import build_surface
from volforecast.surface.plot import plot_smile, plot_term_structure
from volforecast.forecast.validation import run_cv
from volforecast.forecast.metrics import make_results_table
from volforecast.forecast.implied_vs_forecast import compare_chain_to_forecast

TICKER = 'SPY'
START  = '2019-01-01'
END    = '2024-12-31'
R      = 0.045   # approximate risk-free rate
Q      = 0.013   # approximate SPY dividend yield
SEED   = 42
HORIZON = 21     # 21-trading-day (≈1 month) realized-vol forecast

src = YFinanceSource()
print('Setup complete.')

---
## 1 — Volatility Surface

We fetch SPY's option chain across the nearest six expiries, compute implied vol from each
option's mid-price (Newton-Raphson with bisection fallback), and assemble a surface.

Two views are shown:
* **Smile / skew**: IV vs. strike for the nearest expiry.  The downward slope ("skew") reflects
  demand for downside protection — investors pay a premium for OTM puts.
* **Term structure**: near-ATM IV vs. time to expiry.  Upward-sloping term structure is
  common during calm regimes; it inverts when near-term risk is elevated.

In [ ]:
surface = build_surface(TICKER, source=src, r=R, q=Q, option_type='call', max_expiries=6)
print(f'Surface: {len(surface)} data points across {surface["expiry"].nunique()} expiries')
surface.head()

In [ ]:
expiries = sorted(surface['expiry'].unique())
nearest_expiry = expiries[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_smile(surface, expiry=nearest_expiry, option_type='call', ax=axes[0])
plot_term_structure(surface, option_type='call', ax=axes[1])
fig.suptitle(f'{TICKER} Implied Volatility — {nearest_expiry} (nearest expiry)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 2 — Monte Carlo: Convergence & Variance Reduction

We price a 1-month ATM call using geometric Brownian motion, using the near-ATM IV from
the surface as the volatility input.

**Convergence check**: as path count grows, the MC price converges to the Black-Scholes closed
form.  The standard error shrinks as 1/√N — confirming the statistical properties are correct.

**Variance reduction** reduces the number of paths needed for a given accuracy:
* *Antithetic variates*: pair each draw Z with −Z.  The negatively correlated payoffs cancel
  variance without bias.
* *Control variates*: subtract a known-expectation term (discounted stock price) to absorb
  the dominant source of randomness.  For an ATM call the correlation is ≥ 0.9, giving
  variance reduction well above 50%.

In [ ]:
# Use most recent close as spot; near-ATM IV from the surface.
import datetime
today = datetime.date.today()
recent_ohlcv = src.get_ohlcv(TICKER, start=(today - datetime.timedelta(days=10)).isoformat(),
                              end=today.isoformat())
spot = float(recent_ohlcv['Close'].iloc[-1])

# Near-ATM IV: median IV within 1% of spot for the nearest expiry
near_atm = surface[(surface['expiry'] == nearest_expiry) &
                   (surface['moneyness'].between(0.99, 1.01))]
atm_iv = float(near_atm['iv'].median()) if not near_atm.empty else 0.18

S = spot
K = spot          # ATM
T = 1 / 12        # 1-month
sigma = atm_iv

bs_price = bsm_price(S, K, T, R, sigma, q=Q, option_type='call')
print(f'Spot: {S:.2f}  ATM IV: {atm_iv:.1%}  BS price: {bs_price:.4f}')

In [ ]:
# --- Convergence plot ---
path_counts = [500, 2_000, 5_000, 20_000, 50_000, 100_000]
mc_prices, mc_stderrs = [], []
for n in path_counts:
    p, se = mc_price(S, K, T, R, sigma, q=Q, option_type='call', n_paths=n, seed=SEED)
    mc_prices.append(p)
    mc_stderrs.append(se)

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(path_counts, mc_prices, yerr=mc_stderrs, fmt='o-', capsize=4, label='MC price ± stderr')
ax.axhline(bs_price, color='firebrick', linestyle='--', linewidth=1.5, label=f'BS price {bs_price:.4f}')
ax.set_xscale('log')
ax.set_xlabel('Number of paths (log scale)')
ax.set_ylabel('Option price ($)')
ax.set_title(f'{TICKER} ATM 1-month Call — MC Convergence to Black-Scholes')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Variance reduction table ---
N_VR = 100_000
naive_p, naive_se = mc_price(S, K, T, R, sigma, q=Q, n_paths=N_VR, seed=SEED)
naive_var = naive_se**2 * N_VR  # reconstruct sample variance

anti   = mc_price_antithetic(S, K, T, R, sigma, q=Q, n_paths=N_VR, seed=SEED)
cv_res = mc_price_control_variate(S, K, T, R, sigma, q=Q, n_paths=N_VR, seed=SEED)

vr_table = pd.DataFrame([
    {'Method': 'Naive MC',       'Price': naive_p,           'StdErr': naive_se,
     'Sample Variance': naive_var, 'Variance Reduction %': 0.0},
    {'Method': 'Antithetic',     'Price': anti['price'],     'StdErr': anti['stderr'],
     'Sample Variance': anti['var_antithetic'], 'Variance Reduction %': anti['reduction_pct']},
    {'Method': 'Control Variate','Price': cv_res['price'],   'StdErr': cv_res['stderr'],
     'Sample Variance': cv_res['var_cv'], 'Variance Reduction %': cv_res['reduction_pct']},
]).set_index('Method')

print(f'Black-Scholes reference price: {bs_price:.6f}\n')
vr_table.style.format({'Price': '{:.6f}', 'StdErr': '{:.6f}',
                       'Sample Variance': '{:.2e}', 'Variance Reduction %': '{:.1f}'})

---
## 3 — Realized-Vol Forecast: Time Series View

We run expanding-window walk-forward cross-validation on daily SPY OHLCV (2019–2024).
The forecast target is 21-day forward realized volatility (annualized std of log returns).

Three models compete on identical, purged folds:
* **ML** — gradient-boosted trees on lagged-vol features (Parkinson, Garman-Klass, EWMA lags)
* **EWMA** — exponentially weighted moving average of squared daily returns (λ = 0.94)
* **GARCH(1,1)** — refitted in-sample for each fold; single h-step volatility forecast

All predictions below are strictly out-of-sample — the model never saw the test data it
predicted on.  Scaler / transform fitting happens inside each fold.

In [ ]:
ohlcv = src.get_ohlcv(TICKER, start=START, end=END)
print(f'OHLCV loaded: {len(ohlcv)} trading days  ({ohlcv.index[0].date()} → {ohlcv.index[-1].date()})')

print('Running walk-forward CV (this may take a minute) ...')
cv_results = run_cv(ohlcv, horizon=HORIZON, min_train=252, step=21)
n_oos = len(cv_results['ML'])
print(f'CV complete — {n_oos} out-of-sample predictions per model')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ml_df = cv_results['ML'].set_index('date').sort_index()
ax.plot(ml_df.index, ml_df['y_true'] * 100,  color='black',      lw=1.2, label='Realized vol (actual)')
ax.plot(ml_df.index, ml_df['y_pred'] * 100,  color='steelblue',  lw=1.0, alpha=0.85, label='ML forecast')

ew_df = cv_results['EWMA'].set_index('date').sort_index()
ax.plot(ew_df.index, ew_df['y_pred'] * 100,  color='darkorange',  lw=1.0, alpha=0.75, label='EWMA')

g_df = cv_results['GARCH'].set_index('date').sort_index()
ax.plot(g_df.index, g_df['y_pred'] * 100,    color='forestgreen', lw=1.0, alpha=0.75, label='GARCH(1,1)')

ax.set_xlabel('Date')
ax.set_ylabel('Annualized Volatility (%)')
ax.set_title(f'{TICKER} — 21-day Realized Vol: Forecast vs Actual (out-of-sample, 2019–2024)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4 — Error Table vs Baselines

RMSE and MAE are straightforward error metrics on annualized vol.
**QLIKE** (Patton 2011) is the loss function derived from the Gaussian quasi-log-likelihood
and is scale-invariant — preferred for volatility evaluation.
**Corr** is the Pearson correlation between predicted and realized vol.

All metrics are pooled across all out-of-sample folds.

In [ ]:
results_table = make_results_table(cv_results)
print(results_table.to_string(float_format='{:.6f}'.format))
results_table.style.format({
    'RMSE':  '{:.4f}',
    'MAE':   '{:.4f}',
    'QLIKE': '{:.4f}',
    'Corr':  '{:.4f}',
    'N':     '{:,d}',
}).highlight_min(subset=['RMSE', 'MAE', 'QLIKE'], color='lightgreen'
).highlight_max(subset=['Corr'], color='lightgreen')

In [ ]:
# Honest summary: report ranking and whether ML beats GARCH
best_model = results_table.index[0]
ml_rmse    = results_table.loc['ML',      'RMSE']
garch_rmse = results_table.loc['GARCH',   'RMSE']
ewma_rmse  = results_table.loc['EWMA',    'RMSE']

print(f'Best model by RMSE: {best_model}')
if best_model == 'ML':
    print(f'ML beats GARCH by {(garch_rmse - ml_rmse) / garch_rmse:.1%} RMSE — a real but modest gain.')
else:
    print(
        f'ML (RMSE={ml_rmse:.4f}) does not beat {best_model} (RMSE={results_table["RMSE"].min():.4f}).\n'
        'Honest result: GARCH-class baselines are strong for short-horizon equity vol.'
    )
print('\nLimitations: free Yahoo Finance data, survivorship in ticker selection, '
      'no transaction-cost modeling, single ticker only.')

---
## 5 — Implied vs Forecast: Rich/Cheap View

The final step bridges the forecasting model and the options market.

We take the ML model's **most recent out-of-sample forecast** as our view of expected realized
vol over the next month, then compare it to the **market-implied vol** for each strike in the
current nearest-expiry option chain.

**Richness = IV − Forecast RV**
* Positive (red) → market IV exceeds model forecast; option looks *rich* relative to the view
* Negative (blue) → market IV below model forecast; option looks *cheap*

The skew means OTM puts are typically rich (high IV) while OTM calls can be closer to neutral.

> **Note:** this is a research signal only — a stylized comparison between one model's point
> forecast and current market prices.  It does not account for risk premium, transaction costs,
> model uncertainty, or liquidity.

In [ ]:
# Latest out-of-sample ML prediction as the current forecast
latest_ml_df = cv_results['ML'].sort_values('date')
forecast_rv  = float(latest_ml_df['y_pred'].iloc[-1])
forecast_date = latest_ml_df['date'].iloc[-1]
print(f'ML forecast RV (as of {forecast_date.date()}): {forecast_rv:.1%} annualized')

In [ ]:
# Build the richness table for the nearest expiry
chain = src.get_option_chain(TICKER, expiry=nearest_expiry)

richness_df = compare_chain_to_forecast(
    calls=chain['calls'],
    puts=chain['puts'],
    spot=spot,
    expiry=nearest_expiry,
    forecast_rv=forecast_rv,
    r=R,
    q=Q,
    moneyness_range=(0.85, 1.15),
)

print(f'Options in richness table: {len(richness_df)}')
richness_df.head(10)

In [ ]:
if richness_df.empty:
    print('No option data available for richness plot (chain may be expired or unavailable).')
else:
    fig, ax = plt.subplots(figsize=(12, 5))

    for ot, color, marker in [('call', 'steelblue', 'o'), ('put', 'darkorange', 's')]:
        sub = richness_df[richness_df['option_type'] == ot]
        if sub.empty:
            continue
        ax.scatter(
            sub['moneyness'],
            sub['richness'] * 100,
            c=color, marker=marker, s=40, alpha=0.8, label=ot,
        )

    ax.axhline(0, color='black', linewidth=1.2, linestyle='--', label='Breakeven (IV = Forecast)')
    ax.axvline(1.0, color='grey', linewidth=0.8, linestyle=':')

    ax.fill_between([0.85, 1.15], 0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 5,
                    alpha=0.04, color='red', label='Rich region')
    ax.fill_between([0.85, 1.15], ax.get_ylim()[0] if ax.get_ylim()[0] < 0 else -5, 0,
                    alpha=0.04, color='blue', label='Cheap region')

    ax.set_xlabel('Moneyness (K / S)')
    ax.set_ylabel('Richness = IV − Forecast RV (ppts)')
    ax.set_title(
        f'{TICKER} Options — Implied Vol vs ML Forecast ({nearest_expiry})\n'
        f'ML Forecast RV = {forecast_rv:.1%}  |  Spot = {spot:.2f}'
    )
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    n_rich  = (richness_df['richness'] > 0).sum()
    n_cheap = (richness_df['richness'] < 0).sum()
    print(f'Rich options (IV > forecast): {n_rich} | Cheap options (IV < forecast): {n_cheap}')

---
## Summary

| Component | What was shown |
|---|---|
| Vol surface | Persistent downside skew; upward-sloping term structure in calm markets |
| Monte Carlo | Convergence to BS confirmed; control variates cut variance by >50% |
| Forecasting | Walk-forward CV on 6 years of data; GARCH is a strong baseline |
| Rich/cheap | IV skew means OTM puts are typically rich vs. the flat-vol forecast |

**Limitations and next steps**
- Data quality: Yahoo Finance free data; no survivorship-bias correction.
- Single ticker: conclusions may not generalize; vol regimes differ across underlyings.
- No risk premium: the vol risk premium (IV > RV on average) is expected, not alpha.
- Model uncertainty: point forecast used; should propagate forecast confidence intervals.
- Next steps: extend to a cross-sectional panel, add jump-diffusion, model IV directly.